# Knowledge Tracing: per-student topic mastery (BKT)

Fits a **BKT** (Bayesian Knowledge Tracing) model over this project's student interaction
graph to answer the actual product question: *for a given student, which topics are they
weak in?*

BKT is a 2-state HMM per (student, leaf topic), updated online per attempt — cheap,
interpretable, and it needs only a handful of attempts per pair to produce a defensible
`p_know`, including graceful cold-start behavior (falls back to the prior) for a new
student or a just-published topic. Its output is directly the personalization signal
this system needs: one `p_know` per (student, topic), so "what is this student weak in"
is just "sort their topics by `p_know` ascending."

Data: `scripts/generate_dummy_interactions.py`'s synthetic students, generated against
the real question bank's subject/topic taxonomy (`notebooks/mcq_output/question_bank.json`)
so topic-level sequences here have the same shape production data will have. Current scale:
20 students, ~1100 quiz answers, 56 simulated topics.

Requires `make neo4j-up` and the dummy data already generated (see
`docs/07-operations.md` / this repo's README).

In [1]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import roc_auc_score, log_loss

PROJECT_ROOT = Path.cwd().parent  # notebook lives in notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from src.student_kg.driver import make_driver

pd.set_option("display.max_rows", 20)
RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

## 1. Pull interaction sequences from Neo4j

One row per `QUIZ_ANSWER` event, joined to the leaf `Topic` it's tagged with via
`(:Question)-[:BELONGS_TO]->(:Topic)` (see `src/quiz/attempts.py::record_attempt` — the
leaf topic is the deepest element of `topic_tag`). Ordered by `ts` within each student so
sequence models see the real chronological order.

In [2]:
_SEQUENCES_QUERY = """
MATCH (s:Student)-[:ATTEMPTED]->(:QuizSession)-[:HAS_ANSWER]->(e:InteractionEvent {type: "QUIZ_ANSWER"})
MATCH (e)-[:FOR_QUESTION]->(q:Question)-[:BELONGS_TO]->(t:Topic)
OPTIONAL MATCH (s)-[r:REVIEWING]->(q)
RETURN s.id AS student_id, t.path AS topic_path, e.question_uid AS question_uid,
       e.correct AS correct, e.confidence AS confidence, e.ts AS ts,
       r.attempt_count AS attempt_count
ORDER BY s.id, e.ts ASC
"""

driver = make_driver()
with driver.session() as session:
    records = [dict(r) for r in session.run(_SEQUENCES_QUERY)]
driver.close()

df = pd.DataFrame(records)
df["ts"] = df["ts"].apply(lambda z: z.to_native())
df["correct"] = df["correct"].astype(int)
df["confidence"] = df["confidence"].fillna("unsure")
print(f"{len(df)} attempts, {df.student_id.nunique()} students, {df.topic_path.nunique()} topics")
print(df["confidence"].value_counts())
df.head()

10480 attempts, 43 students, 56 topics
confidence
unsure       3682
guessing     3629
confident    3169
Name: count, dtype: int64


,student_id,topic_path,question_uid,correct,confidence,ts,attempt_count
0,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Abdominal wall - hollow organ - organs > Abdom...,master_mcq::copy_of_paket_a12_pdf::0002,1,unsure,2026-03-03 00:03:23.547578+00:00,4
1,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Abdominal wall - hollow organ - organs > Abdom...,master_mcq::copy_of_paket_a12_pdf::0002,0,unsure,2026-03-03 00:04:08.876737+00:00,4
2,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Abdominal wall - hollow organ - organs > Abdom...,master_mcq::copy_of_paket_a12_pdf::0002,0,unsure,2026-03-03 00:07:00.502431+00:00,4
3,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Abdominal wall - hollow organ - organs > Abdom...,master_mcq::copy_of_paket_a12_pdf::0002,0,unsure,2026-03-03 00:09:07.288970+00:00,4
4,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Abdominal wall - hollow organ - organs > Abdom...,master_mcq::paket_27_pdf::0001,0,guessing,2026-03-03 00:12:48.884400+00:00,3


In [3]:
# Data-scale diagnostics up front — these numbers are what make the BKT-vs-AKT call later.
attempts_per_student_topic = df.groupby(["student_id", "topic_path"]).size()
print("attempts per (student, topic) pair:")
print(attempts_per_student_topic.describe())
print()
print("median attempts/student/topic:", attempts_per_student_topic.median())
print("pairs with >=5 attempts:", (attempts_per_student_topic >= 5).sum(),
      "/", len(attempts_per_student_topic))

attempts per (student, topic) pair:
count    1554.000000
mean        6.743887
std         2.266823
min         3.000000
25%         6.000000
50%         6.000000
75%         6.000000
max        24.000000
dtype: float64

median attempts/student/topic: 6.0
pairs with >=5 attempts: 1509 / 1554


## 2. BKT baseline (confidence-conditional emissions)

Standard 2-state (knows / doesn't-know) HMM per (student, topic):

- `p_init` — prior P(knows) before any evidence.
- `p_transit` — P(learns) between an unknown state and the next attempt.
- `p_slip` — P(wrong | knows).
- `p_guess` — P(correct | doesn't know).

Every attempt already carries a self-reported `confidence` label (`confident` / `unsure` /
`guessing` — see `src/quiz/attempts.py::record_attempt`'s docstring: "a correct guess and a
confident correct answer are not the same evidence of mastery"). Plain BKT throws that
signal away and treats every correct/wrong answer as equally strong evidence. Instead we
make `p_slip` and `p_guess` **conditional on the reported confidence**, so the same
correct/wrong outcome updates `p_know` by a different amount depending on how the student
said they got there:

- `confident` — low `p_slip`, low `p_guess`: a confident-correct answer is trusted as real
  evidence of knowing; a confident-wrong answer is trusted as real evidence of a
  misconception (not a slip), so it pulls `p_know` down hard.
- `guessing` — high `p_slip`, high `p_guess`: outcomes are close to a coin flip either way,
  so both correct and wrong answers move `p_know` only a little — a lucky guess shouldn't
  look like mastery, and a wrong guess shouldn't look like a confirmed gap.
- `unsure` — in between the two, close to the original flat defaults.

Update rule (per attempt, params now indexed by that attempt's `confidence`):

```
p_slip, p_guess = P_SLIP[confidence], P_GUESS[confidence]

if observed correct:
    p_know_post = p_know * (1 - p_slip) / (p_know * (1 - p_slip) + (1 - p_know) * p_guess)
else:
    p_know_post = p_know * p_slip / (p_know * p_slip + (1 - p_know) * (1 - p_guess))

p_know_next = p_know_post + (1 - p_know_post) * p_transit
```

`p_init`/`p_transit` stay global scalars — only the emission probabilities (`p_slip`,
`p_guess`) vary by confidence. All params are still fixed defaults (not EM-fit per topic);
EM-fitting per topic, and validating the confidence buckets actually separate accuracy the
way we're assuming (guessing ≈ chance rate, confident ≫ unsure), is follow-up work.

In [4]:
BKT_PARAMS = dict(
    p_init=0.3,
    p_transit=0.1,
    p_slip={"confident": 0.05, "unsure": 0.10, "guessing": 0.30},
    p_guess={"confident": 0.10, "unsure": 0.25, "guessing": 0.50},
)


def bkt_update(p_know: float, correct: bool, confidence: str, params: dict) -> float:
    p_slip = params["p_slip"][confidence]
    p_guess = params["p_guess"][confidence]
    p_transit = params["p_transit"]
    if correct:
        num = p_know * (1 - p_slip)
        denom = num + (1 - p_know) * p_guess
    else:
        num = p_know * p_slip
        denom = num + (1 - p_know) * (1 - p_guess)
    p_know_post = num / denom if denom > 0 else p_know
    return p_know_post + (1 - p_know_post) * p_transit


def bkt_predict_proba(p_know: float, confidence: str, params: dict) -> float:
    """P(correct) implied by current p_know, before seeing the observation. Needs
    `confidence` too since p_slip/p_guess are now confidence-conditional — this is a
    genuine next-step prediction only in hindsight-eval mode, where the label is known
    ahead of time from logged data; a live recommender doesn't know the student's
    confidence before they answer, so it would predict off the `unsure` (middle) row
    instead."""
    p_slip = params["p_slip"][confidence]
    p_guess = params["p_guess"][confidence]
    return p_know * (1 - p_slip) + (1 - p_know) * p_guess


def run_bkt(df: pd.DataFrame, params: dict = BKT_PARAMS) -> tuple[pd.DataFrame, dict]:
    """Runs BKT forward over every (student, topic) sequence in chronological order.
    Returns per-attempt predicted P(correct) (predicted BEFORE the update, i.e. a genuine
    next-step prediction, not a fitted-in-hindsight one) and the final p_know per pair."""
    preds = np.empty(len(df))
    state: dict[tuple[str, str], float] = {}
    for i, row in enumerate(df.itertuples()):
        key = (row.student_id, row.topic_path)
        p_know = state.get(key, params["p_init"])
        preds[i] = bkt_predict_proba(p_know, row.confidence, params)
        state[key] = bkt_update(p_know, bool(row.correct), row.confidence, params)
    out = df.copy()
    out["bkt_pred"] = preds
    return out, state


df_sorted = df.sort_values(["student_id", "topic_path", "ts"]).reset_index(drop=True)
bkt_results, bkt_final_state = run_bkt(df_sorted)
bkt_results[["student_id", "topic_path", "ts", "correct", "confidence", "bkt_pred"]].head(10)

,student_id,topic_path,ts,correct,confidence,bkt_pred
0,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Abdominal wall - hollow organ - organs > Abdom...,2026-03-03 00:03:23.547578+00:00,1,unsure,0.445000
1,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Abdominal wall - hollow organ - organs > Abdom...,2026-03-03 00:04:08.876737+00:00,0,unsure,0.669944
2,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Abdominal wall - hollow organ - organs > Abdom...,2026-03-03 00:07:00.502431+00:00,0,unsure,0.429511
3,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Abdominal wall - hollow organ - organs > Abdom...,2026-03-03 00:09:07.288970+00:00,0,unsure,0.343319
4,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Abdominal wall - hollow organ - organs > Abdom...,2026-03-03 00:12:48.884400+00:00,0,guessing,0.523935
5,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Abdominal wall - hollow organ - organs > Abdom...,2026-03-03 00:16:13.070767+00:00,1,unsure,0.359118
6,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Abdominal wall - hollow organ - organs > Abdom...,2026-06-25 12:25:24.724696+00:00,1,unsure,0.561119
7,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Abdominal wall - hollow organ - organs > Abdom...,2026-06-25 12:26:50.021167+00:00,1,guessing,0.658189
8,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Abdominal wall - hollow organ - organs > Abdom...,2026-06-25 12:29:24.805699+00:00,1,unsure,0.807096
9,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Abdominal wall - hollow organ - organs > Abdom...,2026-06-25 12:32:17.265653+00:00,1,unsure,0.874100


In [5]:
bkt_auc = roc_auc_score(bkt_results["correct"], bkt_results["bkt_pred"])
bkt_ll = log_loss(bkt_results["correct"], bkt_results["bkt_pred"].clip(1e-4, 1 - 1e-4))
print(f"BKT next-step prediction — AUC: {bkt_auc:.4f}, log loss: {bkt_ll:.4f}")

BKT next-step prediction — AUC: 0.5658, log loss: 0.7491


In [6]:
# Sanity check on the P_SLIP/P_GUESS assumption baked into BKT_PARAMS: raw accuracy
# should actually separate by confidence bucket (guessing near chance rate, confident
# well above it) before trusting the conditional emission model over the flat one.
print(bkt_results.groupby("confidence")["correct"].agg(["mean", "count"]))

                mean  count
confidence                 
confident   0.712212   3169
guessing    0.326261   3629
unsure      0.534221   3682


## 3. Per-student weak-topic view

This is the shape a `MASTERS` edge (`Student -> Topic`, `p_know`) would take in the graph
— see the design note at the end for wiring this into `src/quiz/attempts.py::record_attempt`.

Each student gets their own ranked list — this is the personalization output: not one
global model output, but one `p_know` value per (student, topic) pair, so weak topics are
specific to that student's own history and never averaged across the cohort.

In [7]:
mastery_rows = [
    {"student_id": sid, "topic_path": tp, "p_know": pk}
    for (sid, tp), pk in bkt_final_state.items()
]
mastery_df = pd.DataFrame(mastery_rows)

n_observations = (
    df_sorted.groupby(["student_id", "topic_path"]).size().rename("n_observations")
)
mastery_df = mastery_df.join(n_observations, on=["student_id", "topic_path"])

# attempt_count (from the :REVIEWING edge, src/quiz/attempts.py::record_attempt) is a
# lifetime per-question retry counter — it never resets, unlike streak. Averaged per
# (student, topic) it flags a pattern plain accuracy/n_observations can't: a topic where
# the student keeps re-attempting the same questions and still isn't landing a strong
# pass reads as "stuck", not just "weak" or "under-observed".
avg_attempt_count = (
    df_sorted.groupby(["student_id", "topic_path"])["attempt_count"]
    .mean()
    .rename("avg_attempt_count")
)
mastery_df = mastery_df.join(avg_attempt_count, on=["student_id", "topic_path"])
mastery_df.sort_values(["student_id", "p_know"]).head(10)

,student_id,topic_path,p_know,n_observations,avg_attempt_count
36,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Thyroid > Thyroid hormone synthesis,0.111422,6,1.333333
32,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Histology - DIgestive 1 (2026) > Liver functio...,0.124054,6,1.333333
7,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Anatomy of Neck > Cervical nerve anatomy,0.136261,20,5.100000
1,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Abdominal wall - hollow organ - organs > Liver...,0.170192,11,3.000000
19,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Embriologi Pencernaan > embriologi pencernaan,0.247405,12,2.333333
2,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Anatomy of Endocrine Glands > Adrenal gland in...,0.254674,6,1.333333
33,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Histology - Endocrine (2026) > Renal tubule hi...,0.254674,6,1.666667
13,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,ENZIME FK 02 2024 rc > Kinetika Enzim Michaeli...,0.293939,6,1.666667
22,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Endocrine 1-25 > Oxytocin release mechanism,0.303741,6,1.333333
20,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Endocrine 1-25 > Antidiuretic hormone function,0.305861,6,2.333333


In [8]:
def weakest_topics(
    student_id: str,
    mastery_df: pd.DataFrame,
    top_n: int = 5,
    min_observations: int = 3,
    stuck_attempt_threshold: float = 2.0,
) -> pd.DataFrame:
    """This student's lowest-p_know topics, ascending — the recommend-what-to-study query
    the personalization system runs (GET /students/me/mastery, sorted client- or
    server-side).

    Adds a `low_evidence` flag for pairs with fewer than `min_observations` attempts: with
    only 1-2 attempts, p_know is still close to p_init (the prior) rather than a real
    read on the student, so surfacing it as "weak" without qualification would overstate
    confidence. Rows are NOT dropped — a topic never attempted is still worth surfacing as
    "unknown, go try it" — just labeled so the caller (or UI) can render it differently
    (e.g. "not enough data yet" instead of a confident weak-topic claim).

    Also adds a `stuck` flag for pairs with enough evidence (not low_evidence) whose
    avg_attempt_count is at least `stuck_attempt_threshold` — the student has repeatedly
    re-attempted these questions (not just answered once and moved on) and still hasn't
    built up p_know, which reads differently from a topic that's merely weak on first
    exposure and worth calling out separately in the UI (e.g. "revisit with a different
    approach" instead of "just practice more")."""
    out = (
        mastery_df[mastery_df["student_id"] == student_id]
        .sort_values("p_know")
        .head(top_n)
        .reset_index(drop=True)
    )
    out["low_evidence"] = out["n_observations"] < min_observations
    out["stuck"] = (~out["low_evidence"]) & (
        out["avg_attempt_count"] >= stuck_attempt_threshold
    )
    return out


example_student = mastery_df["student_id"].iloc[0]
print(f"weakest topics for student {example_student}:")
weakest_topics(example_student, mastery_df)

weakest topics for student 05bbbea4-5b54-4fae-8ab3-6abce681c0d1:


,student_id,topic_path,p_know,n_observations,avg_attempt_count,low_evidence,stuck
0,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Thyroid > Thyroid hormone synthesis,0.111422,6,1.333333,False,False
1,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Histology - DIgestive 1 (2026) > Liver functio...,0.124054,6,1.333333,False,False
2,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Anatomy of Neck > Cervical nerve anatomy,0.136261,20,5.100000,False,True
3,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Abdominal wall - hollow organ - organs > Liver...,0.170192,11,3.000000,False,True
4,05bbbea4-5b54-4fae-8ab3-6abce681c0d1,Embriologi Pencernaan > embriologi pencernaan,0.247405,12,2.333333,False,True


In [9]:
sid = example_student
weak = weakest_topics(sid, mastery_df, top_n=3)
for tp in weak["topic_path"]:
    sub = df_sorted[(df_sorted.student_id == sid) & (df_sorted.topic_path == tp)]
    print(tp, "acc:", sub["correct"].mean(), "n:", len(sub))

Thyroid > Thyroid hormone synthesis acc: 0.5 n: 6
Histology - DIgestive 1 (2026) > Liver functions and structure acc: 0.16666666666666666 n: 6
Anatomy of Neck > Cervical nerve anatomy acc: 0.2 n: 20
